# Data Quality Analysis

Цей ноутбук виконує аналіз якості даних:
- Підключається до БД MySQL
- Зчитує дані з таблиці `documents`
- Рахує пропуски, дублікати
- Перевіряє типи та коректність значень
- Зберігає звіт у `/shared/reports/data_quality_report.json`

In [1]:
import os
import json
import pandas as pd
from sqlalchemy import create_engine

db_host = os.environ.get('MYSQL_HOST', 'db')
db_user = os.environ.get('MYSQL_USER', 'appuser')
db_password = os.environ.get('MYSQL_PASSWORD', 'apppassword')
db_name = os.environ.get('MYSQL_DATABASE', 'docflow')

connection_string = f'mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/{db_name}'
engine = create_engine(connection_string)
print('[data_quality_analysis] Підключення до БД встановлено.')

ModuleNotFoundError: No module named 'pymysql'

In [ ]:
df = pd.read_sql('SELECT * FROM documents', engine)
print(f'Завантажено {len(df)} записів')
print(f'Стовпці: {list(df.columns)}')
df.head()

In [ ]:
missing_values = df.isnull().sum().to_dict()
missing_percent = (df.isnull().sum() / len(df) * 100).round(2).to_dict()

print('=== Пропущені значення ===')
for col, count in missing_values.items():
    print(f'  {col}: {count} ({missing_percent[col]}%)')

In [ ]:
total_duplicates = int(df.duplicated().sum())
print(f'\n=== Дублікати ===')
print(f'Кількість повних дублікатів: {total_duplicates}')

column_duplicates = {}
for col in df.columns:
    dup_count = int(df[col].duplicated().sum())
    column_duplicates[col] = dup_count
    print(f'  Дублікати у "{col}": {dup_count}')

In [ ]:
data_types = {col: str(dtype) for col, dtype in df.dtypes.items()}
print('\n=== Типи даних ===')
for col, dtype in data_types.items():
    print(f'  {col}: {dtype}')

total_rows = len(df)
total_columns = len(df.columns)
memory_usage = float(df.memory_usage(deep=True).sum() / 1024)

print(f'\nЗагальна кількість рядків: {total_rows}')
print(f'Загальна кількість стовпців: {total_columns}')
print(f'Використання памʼяті: {memory_usage:.2f} KB')

In [ ]:
empty_strings = {}
for col in df.select_dtypes(include='object').columns:
    empty_count = int((df[col].str.strip() == '').sum())
    empty_strings[col] = empty_count

print('\n=== Порожні рядки ===')
for col, count in empty_strings.items():
    print(f'  {col}: {count}')

In [ ]:
report = {
    'total_rows': total_rows,
    'total_columns': total_columns,
    'memory_usage_kb': round(memory_usage, 2),
    'data_types': data_types,
    'missing_values': missing_values,
    'missing_percent': missing_percent,
    'total_duplicates': total_duplicates,
    'column_duplicates': column_duplicates,
    'empty_strings': empty_strings,
    'columns': list(df.columns)
}

report_path = '/shared/reports/data_quality_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f'\nЗвіт збережено: {report_path}')
print(json.dumps(report, ensure_ascii=False, indent=2))